# L38 · FFT 蝶形分治（divide and conquer） — 偶奇拆分、蝶形运算 `E[k]+W^k·O[k]`，O(N²)→O(N log N)

**目标**：理解 Cooley-Tukey 分治——偶数下标 DFT + 奇数下标 DFT + 蝶形合并 = N 点 DFT，复杂度从 `O(N²)` 降到 `O(N log N)`。

🔗 **Aurora 连接**：`aurora.audio.transforms._fft_recursive()` 就是本节的递归核心；实现 `butterfly()` 后你将理解那个函数的每一行。

DFT 朴素实现对每个频率 `k` 都要遍历全部 `N` 个样本，共 `N²` 次乘法。Cooley-Tukey 的洞察是：偶数下标和奇数下标的 DFT 各只有 `N/2` 点，把两个半结果用一次蝶形合并就能得到完整 N 点结果。这个分治可以递归到 `N=1` 为止，总层数 `log₂N`，每层 `N/2` 个蝶形——乘法次数变成 `(N/2) log₂N`。

In [ ]:
import numpy as np

## 定位图：N=8 蝶形网络全貌

先看整体结构，再读公式推导。图中每条彩色 **X** 是一个蝶形单元；三层颜色对应三次递归合并（红→蓝→绿）；左侧输入按**位反转（bit reversal）序**排列，右侧输出为自然序频谱 X[0..7]。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(11, 5.5))

N = 8
n_stages = int(np.log2(N))  # = 3

xs = [0.0, 1.8, 3.6, 5.4]                  # x-coord for each column
ys = np.linspace(1.0, 0.0, N)              # y-coord: index 0 at top
COLORS = ['#e74c3c', '#3498db', '#2ecc71'] # red / blue / green per stage

# column headers
for x, lbl in zip(xs, ['输入\n(位反转序)', '第 1 层\n步长 1', '第 2 层\n步长 2', '第 3 层\n步长 4']):
    ax.text(x, 1.12, lbl, ha='center', va='bottom', fontsize=8.5,
            fontweight='bold', color='#2c3e50')

# butterfly connections
for stage in range(n_stages):
    stride = 2 ** stage
    color  = COLORS[stage]
    x_l, x_r = xs[stage], xs[stage + 1]
    for group in range(0, N, 2 * stride):
        for k in range(stride):
            u, v   = group + k, group + k + stride
            yu, yv = ys[u], ys[v]
            ax.plot([x_l, x_r], [yu, yu], '-', color='#bdc3c7', lw=1.2, zorder=1)
            ax.plot([x_l, x_r], [yv, yv], '-', color='#bdc3c7', lw=1.2, zorder=1)
            ax.plot([x_l, x_r], [yu, yv], '-', color=color, lw=2.2, alpha=0.72, zorder=2)
            ax.plot([x_l, x_r], [yv, yu], '-', color=color, lw=2.2, alpha=0.72, zorder=2)

# nodes
for x in xs:
    for i in range(N):
        ax.plot(x, ys[i], 'o', color='white', ms=9, mec='#2c3e50', mew=1.8, zorder=5)

# input (bit-reversed) / output labels
for i, br in enumerate([0, 4, 2, 6, 1, 5, 3, 7]):
    ax.text(xs[0]  - 0.18, ys[i], f'x[{br}]', ha='right', va='center', fontsize=8.5)
    ax.text(xs[-1] + 0.18, ys[i], f'X[{i}]',  ha='left',  va='center', fontsize=8.5)

# twiddle-factor labels on last stage
x_tw = (xs[2] + xs[3]) / 2
for k in range(4):
    y_mid = (ys[k] + ys[k + 4]) / 2
    ax.text(x_tw, y_mid, f'$W_8^{k}$', ha='center', va='center',
            fontsize=8, color=COLORS[2],
            bbox=dict(fc='white', ec='none', alpha=0.85, pad=1))

# legend
ax.legend(
    handles=[mpatches.Patch(color=c, alpha=0.8, label=f'第 {i+1} 层  步长 {2**i}')
             for i, c in enumerate(COLORS)],
    loc='lower right', fontsize=8.5, framealpha=0.9, edgecolor='#d0d3d4',
)

ax.set_xlim(-0.95, 6.25)
ax.set_ylim(-0.08, 1.25)
ax.axis('off')
ax.set_title('N = 8  Cooley-Tukey DIT-FFT 蝶形信号流图\n'
             '彩色 "X" 为蝶形单元；三种颜色对应三层递归合并',
             fontsize=10.5, fontweight='bold', pad=4, color='#2c3e50')
plt.tight_layout()
plt.show()

## 1. 分治公式：偶奇拆分 + 蝶形合并

设 `x` 长度为 `N`，偶数下标子序列 `x_e = x[0,2,4,...]`，奇数下标子序列 `x_o = x[1,3,5,...]`，各长 `N/2`。令：

```
E[k] = DFT(x_e)[k]    k = 0..N/2-1
O[k] = DFT(x_o)[k]    k = 0..N/2-1
W_N^k = exp(-2πi·k/N)       # 旋转因子（twiddle factor）
```

则完整 N 点 DFT 为：

```
X[k]       = E[k] + W_N^k · O[k]      k = 0..N/2-1
X[k + N/2] = E[k] - W_N^k · O[k]      k = 0..N/2-1
```

上半 `X[0..N/2-1]` 加旋转因子，下半 `X[N/2..N-1]` 减——因为 `W_N^(k+N/2) = -W_N^k`，符号自动翻转，不需要额外计算。

In [ ]:
# 验证分治公式：手动对 N=4 信号拆偶奇再合并
x = np.array([1, 2, 3, 4], dtype=complex)
N = len(x)

x_e = x[::2]   # [1, 3]
x_o = x[1::2]  # [2, 4]

E = np.fft.fft(x_e)  # 2点DFT
O = np.fft.fft(x_o)  # 2点DFT

k = np.arange(N // 2)
twiddles = np.exp(-2j * np.pi * k / N)  # W_N^k

X_top    = E + twiddles * O
X_bottom = E - twiddles * O
X_manual = np.concatenate([X_top, X_bottom])

X_ref = np.fft.fft(x)
print('手动分治结果 :', np.round(X_manual, 4))
print('numpy.fft 结果:', np.round(X_ref,    4))
print('完全吻合     :', np.allclose(X_manual, X_ref))

## 2. 一次蝶形：2 输入 → 2 输出，复杂度为何是 O(N log N)

**蝶形（butterfly）** 指对一对复数 `(a, b)` 做：

```
top    = a + W · b
bottom = a - W · b
```

只需 1 次复数乘（`W·b`）和 2 次复数加减，输出 2 个结果。画在信号流图（signal flow graph）上，两条线交叉如蝴蝶翅膀，名字由此而来。

N 点 FFT 的层次结构：

```
层数    = log₂N
每层蝶形 = N/2 个
总蝶形  = (N/2) · log₂N
```

对比朴素 DFT 的 `N²` 次乘法：`N=1024` 时 FFT 只需约 5120 次，朴素需要 1048576 次，快了 200 倍。

In [ ]:
# 比较朴素 DFT 与 FFT 的操作数（理论值）
for n_exp in [4, 8, 10, 12]:
    N = 2 ** n_exp
    naive_mults = N * N
    fft_butterflies = (N // 2) * n_exp  # 每个蝶形1次乘
    ratio = naive_mults / fft_butterflies
    print(f'N={N:5d}: 朴素={naive_mults:8d}  FFT蝶形={fft_butterflies:5d}  加速比={ratio:.1f}x')

## 3. 终止条件：N=1 时 DFT(x) = x

递归的最底层：当序列只有 1 个样本 `x[0]` 时，DFT 定义为：

```
X[0] = sum_{n=0}^{0} x[n] · exp(-2πi·0·0/1) = x[0] · 1 = x[0]
```

单点信号没有任何频率结构，DFT 输出就是输入本身。这是递归的基础情形，不需要任何乘法。

每层向上合并时，两个 `N/2` 点的结果通过蝶形合并成 `N` 点结果，直到最顶层得到完整 `X[0..N-1]`。

In [ ]:
# 演示递归：手写极简递归 FFT（仅教学用，不优化）
def fft_recursive_demo(x):
    N = len(x)
    if N == 1:
        return x.copy()  # 终止条件
    E = fft_recursive_demo(x[::2])   # 偶数下标
    O = fft_recursive_demo(x[1::2])  # 奇数下标
    k = np.arange(N // 2)
    W = np.exp(-2j * np.pi * k / N)
    return np.concatenate([E + W * O, E - W * O])

x_test = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=complex)
result = fft_recursive_demo(x_test)
print('递归FFT  :', np.round(result, 3))
print('numpy.fft:', np.round(np.fft.fft(x_test), 3))
print('吻合     :', np.allclose(result, np.fft.fft(x_test)))

## 4. ✏️ 实现 `butterfly(E, O)`

**推理路线**：
1. `N = len(E) * 2`；旋转因子 `twiddles = exp(-2πi · arange(N/2) / N)`
2. `top = E + twiddles * O`（上半频谱 `X[0..N/2-1]`）
3. `bottom = E - twiddles * O`（下半频谱 `X[N/2..N-1]`）
4. 返回 `np.concatenate([top, bottom])`，shape `(N,)`

**参考输入输出**：
```python
E = np.fft.fft(np.array([1, 3], dtype=complex))  # 2点DFT of 偶数下标
O = np.fft.fft(np.array([2, 4], dtype=complex))  # 2点DFT of 奇数下标
# butterfly(E, O) ≈ np.fft.fft([1, 2, 3, 4])
# 即 [10+0j, -2+2j, -2+0j, -2-2j]
```

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def butterfly(E: np.ndarray, O: np.ndarray) -> np.ndarray:
    """单层蝶形合并：两个 N/2 点 DFT → 一个 N 点 DFT。

    参数
    ----
    E : shape (N/2,) complex — 偶数下标子序列的 DFT
    O : shape (N/2,) complex — 奇数下标子序列的 DFT

    返回
    ----
    shape (N,) complex — 完整 N 点 DFT
    """
    half = len(E)
    N = half * 2

    # ✏️ TODO: 计算旋转因子 twiddles，shape (N/2,)
    twiddles = ...

    # ✏️ TODO: 蝶形上半：top = E + twiddles * O
    top = ...

    # ✏️ TODO: 蝶形下半：bottom = E - twiddles * O
    bottom = ...

    # ✏️ TODO: 拼接并返回，shape (N,)
    return ...

In [ ]:
# 检查：butterfly 输出应与 numpy.fft.fft 完全吻合
import numpy as np
E = np.fft.fft(np.array([1, 3], dtype=complex))
O = np.fft.fft(np.array([2, 4], dtype=complex))
try:
    result = butterfly(E, O)
    if result is None or result is ...:
        print("⬜ 请先实现 butterfly 的各 TODO 项")
    else:
        ref = np.fft.fft(np.array([1, 2, 3, 4], dtype=complex))
        assert np.allclose(result, ref, atol=1e-10), f"不匹配：{result} vs {ref}"
        print('butterfly 输出:', np.round(result, 4))
        print('numpy 参考    :', np.round(ref,    4))
        print('✅ butterfly 实现正确，误差 < 1e-10')
except TypeError:
    print("⬜ 请先替换 ... 占位符后再运行检查格")


## 5. 参数实验：手算 N=4 蝶形网络

N=4 的 Cooley-Tukey FFT 有 **2 层**，每层 **2 个蝶形**，共 4 次旋转因子乘法。

- **第 1 层**（N=2 子问题）：旋转因子 `W_2^0 = 1`，每对输入只做加减，实际无需复数乘
- **第 2 层**（合并到 N=4）：旋转因子 `W_4^0 = 1`，`W_4^1 = -i`

朴素 DFT 需要 `4² = 16` 次复数乘；FFT 只需 `(4/2)·log₂4 = 4` 次。

**实验**：改变 `N`（必须是 2 的幂），观察 `butterfly` 能否组合成完整 FFT，验证层数与蝶形数。

---
→ **完成 `butterfly` 实现后，打开 [L42 · FFT 图形化](L42_visual_fft.ipynb)，对照蝴蝶图与各类信号的频谱形态，再回来检验你的 `butterfly` 输出。**

In [ ]:
# 用 butterfly 组装两层 FFT（N=4）并数旋转因子乘法次数
mult_count = [0]  # 用列表以便在闭包里修改

def butterfly_counted(E, O):
    """带计数的蝶形：统计旋转因子乘法次数。"""
    half = len(E)
    N_local = half * 2
    k = np.arange(half)
    twiddles = np.exp(-2j * np.pi * k / N_local)
    mult_count[0] += half  # 每个蝶形 half 次乘
    return np.concatenate([E + twiddles * O, E - twiddles * O])

for n_exp in [2, 3, 4, 5]:
    N = 2 ** n_exp
    mult_count[0] = 0
    x = np.random.randn(N) + 1j * np.random.randn(N)

    # 递归 FFT，使用计数版蝶形
    def fft_count(x):
        if len(x) == 1:
            return x.copy()
        E = fft_count(x[::2])
        O = fft_count(x[1::2])
        return butterfly_counted(E, O)

    result = fft_count(x)
    theory = (N // 2) * n_exp
    match = np.allclose(result, np.fft.fft(x))
    print(f'N={N:3d}: 实际旋转乘法={mult_count[0]:4d}  理论(N/2·log₂N)={theory:4d}  FFT正确={match}')

## 本课收束

`butterfly(E, O)` 接受两个 `N/2` 点复数 DFT，通过旋转因子 `W_N^k = exp(-2πi·k/N)` 做加减合并，输出 shape `(N,)` 的完整频谱向量。每次调用只用 `N/2` 次复数乘法，递归展开 `log₂N` 层后总复杂度为 `O(N log N)`。这正是 `aurora.audio.transforms._fft_recursive()` 的核心：每一层递归都调用一次 `butterfly`，终止条件 `N=1` 返回原值。下一课（L39）将从零实现 Cooley-Tukey 递归 FFT，调用 `butterfly` 完成完整的 N 点 FFT，并验证与 `np.fft.fft` 的误差 < 1e-9。更新。